In [51]:
import pandas as pd
import numpy as np
import os

In [55]:
data_to_process = "validation" # "training" # or

In [56]:
files_list = []
for file_name in os.listdir("all_bands_extraction"):
    if file_name.endswith(".csv") and data_to_process in file_name:
        sort_order = file_name.replace(".csv","").split("_")[-1]
        files_list.append((int(sort_order), file_name))

In [57]:
sorted_files_list = sorted_data = sorted(files_list, key=lambda x: x[0])
sorted_files_list

[(200, 'landsat_features_validation_all_bands_0_200.csv')]

In [58]:
df_combined = None
for sort_order, file_name in sorted_files_list:
    if df_combined is None:
        df_combined = pd.read_csv(f"all_bands_extraction/{file_name}")
    else:
        new_df = pd.read_csv(f"all_bands_extraction/{file_name}")
        df_combined = pd.concat((df_combined, new_df), axis=0, ignore_index=True)

In [59]:
for col in df_combined.columns:
    if "Unnamed" in col:
        df_combined.drop(columns=[col], inplace=True)

In [60]:
df_combined

,Latitude,Longitude,Sample Date,lwir,green,cdist,blue,swir22,drad,atmos_opacity,urad,trad,emis,emsd,nir08,atran,red,swir16
0,-32.043333,27.822778,01-09-2014,NaN,12868.0,NaN,11282.0,12421.0,441.0,NaN,821.0,7226.5,9704.0,39.0,15229.0,8729.0,13210.0,14797.0
1,-33.329167,26.077500,16-09-2015,NaN,NaN,-9999.0,NaN,NaN,-4676.5,-9999.0,-4355.0,-702.0,-9999.0,-9999.0,NaN,-827.5,NaN,NaN
2,-32.991639,27.640028,07-05-2015,NaN,9304.5,955.5,8111.5,9958.0,653.0,NaN,1302.0,9005.5,9837.0,47.0,16221.0,8352.0,9145.0,12536.5
3,-34.096389,24.439167,07-02-2012,NaN,NaN,-9999.0,NaN,NaN,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,NaN,-9999.0,NaN,NaN
4,-32.000556,28.581667,01-10-2014,NaN,11100.5,738.5,9504.0,8711.0,778.0,NaN,1577.5,9074.5,9824.0,NaN,9125.0,7928.5,11166.0,9455.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,-33.771111,25.386667,06-12-2012,47168.5,9492.0,142.5,8542.0,10235.0,1554.0,10.0,3366.0,9748.0,9833.0,96.0,17562.0,5979.0,9201.0,13559.5
196,-33.185361,27.390750,04-09-2014,NaN,9083.5,1960.5,8045.0,9484.0,608.0,NaN,1180.0,8432.5,9718.0,80.0,15883.0,8285.0,8931.5,12135.5
197,-32.043333,27.822778,28-09-2015,NaN,10046.5,2035.0,8825.5,10969.0,531.0,NaN,1023.0,9694.5,9706.0,39.0,13619.5,8741.0,10108.0,13105.0
198,-33.001667,25.161389,08-01-2015,NaN,10670.0,7281.5,9505.0,14835.5,1268.0,NaN,2705.0,10848.0,9706.0,52.0,13955.5,6837.0,11638.0,17303.5


In [61]:
original_data = pd.read_csv(f"../landsat_features_{data_to_process}.csv")

In [62]:
# Validate Data is in the same order as the provided training dataset
for index, row in original_data.iterrows():
    print(index)
    lat_orig = row["Latitude"]
    lon_orig = row["Longitude"]
    sample_date_orig = row["Sample Date"]
    green_org = row["green"]

    df_combined_row = df_combined.iloc[index]
    lat_combined = row["Latitude"]
    lon_combined = row["Longitude"]
    sample_date_combined = row["Sample Date"]
    green_combined = row["green"]

    print(f"\tLatitude (orig): {lat_orig}, Latitude (combined): {lat_orig}")
    print(f"\tLongitude (orig): {lon_orig}, Longitude (combined): {lon_orig}")
    print(f"\tSample Date (orig): {sample_date_orig}, Sample Date (combined): {sample_date_combined}")
    print(f"\tGreen (orig): {green_org}, Green (combined): {green_org}")

    assert lat_orig == lat_combined
    assert lon_orig == lon_combined
    assert sample_date_orig == sample_date_combined
    assert np.isnan(green_org) and np.isnan(green_combined) if np.isnan(green_org) else green_org == green_combined

0
	Latitude (orig): -32.043333, Latitude (combined): -32.043333
	Longitude (orig): 27.822778, Longitude (combined): 27.822778
	Sample Date (orig): 01-09-2014, Sample Date (combined): 01-09-2014
	Green (orig): 12868.0, Green (combined): 12868.0
1
	Latitude (orig): -33.329167, Latitude (combined): -33.329167
	Longitude (orig): 26.0775, Longitude (combined): 26.0775
	Sample Date (orig): 16-09-2015, Sample Date (combined): 16-09-2015
	Green (orig): nan, Green (combined): nan
2
	Latitude (orig): -32.99163889, Latitude (combined): -32.99163889
	Longitude (orig): 27.64002778, Longitude (combined): 27.64002778
	Sample Date (orig): 07-05-2015, Sample Date (combined): 07-05-2015
	Green (orig): 9304.5, Green (combined): 9304.5
3
	Latitude (orig): -34.096389, Latitude (combined): -34.096389
	Longitude (orig): 24.439167, Longitude (combined): 24.439167
	Sample Date (orig): 07-02-2012, Sample Date (combined): 07-02-2012
	Green (orig): nan, Green (combined): nan
4
	Latitude (orig): -32.00055556, Lati

In [63]:
df_combined.to_csv(f"landsat_features_{data_to_process}_ross_reprocessed.csv", index=False)